# Detekcija zajednica u društvenim mrežama

**Poglavlje:** [6. Analiza zajednica u društvenim mrežama](../content/06_analiza_zajednica.md)

U ovoj bilježnici **korak po korak** primjenjujemo algoritme za detekciju zajednica na dva primjera:
1. **Mali graf** (9 čvorova, 3 grupe) — da intuitivno vidimo što algoritmi rade.
2. **Zacharyjev karate klub** (34 čvora) — klasičan benchmark u SNA-u.

Za svaki algoritam objašnjavamo logiku, pokrećemo ga, vizualiziramo rezultat i računamo **modularnost**.

- **Definicije** (zajednica, modularnost, algoritmi): [sadržaj pogl. 6 — Definicije](../content/06_analiza_zajednica.md#definicije)
- **Kako radi Louvain / Girvan–Newman**: [sadržaj pogl. 6](../content/06_analiza_zajednica.md#kako-radi-louvain--korak-po-korak)
- **Konceptualni primjeri** (GoT, MCU, glazba, online): [sadržaj pogl. 6 — Primjeri](../content/06_analiza_zajednica.md#primjeri-i-interpretacija)

---

## Korak 1: Uvoz biblioteka i pomoćne funkcije

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from collections import Counter

PALETTE = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabebe", "#469990",
]

def draw_communities(G, partition, pos, title="", ax=None):
    """Crtaj graf s čvorovima obojenima prema zajednici."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    colors = [PALETTE[partition[n] % len(PALETTE)] for n in G.nodes()]
    nx.draw(
        G, pos, ax=ax, node_color=colors, node_size=500,
        with_labels=True, font_size=9, font_weight="bold",
        edge_color="#cccccc", width=1.2,
    )
    unique = sorted(set(partition.values()))
    patches = [mpatches.Patch(color=PALETTE[c % len(PALETTE)], label=f"Zajednica {c}")
               for c in unique]
    ax.legend(handles=patches, loc="upper left", fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

print("Biblioteke učitane.")

---

## Korak 2: Izgradnja malog primjer-grafa (3 zajednice)

Gradimo graf s **9 čvorova** i **3 jasne grupe** (po 3 čvora u svakoj, potpuno povezani unutar grupe). Između grupa dodajemo po jednu vezu — "most". Ovaj graf je dovoljno malen da svaki algoritam možemo pratiti "na ruke".

*(U [sadržaju pogl. 6](../content/06_analiza_zajednica.md) ista ideja ilustrirana je primjerima: GoT kuće, MCU timovi, glazbeni žanrovi.)*

In [ ]:
G_small = nx.Graph()

# Zajednica 1: trokut A-B-C
G_small.add_edges_from([("A", "B"), ("A", "C"), ("B", "C")])
# Zajednica 2: trokut D-E-F
G_small.add_edges_from([("D", "E"), ("D", "F"), ("E", "F")])
# Zajednica 3: trokut G-H-I
G_small.add_edges_from([("G", "H"), ("G", "I"), ("H", "I")])

# Mostovi između zajednica
G_small.add_edges_from([("C", "D"), ("F", "G")])

pos_small = {
    "A": (-2, 1), "B": (-3, 0), "C": (-1, 0),
    "D": (0, 1),  "E": (-1, 2), "F": (1, 2),
    "G": (2, 1),  "H": (3, 0),  "I": (1, 0),
}

# "Stvarne" zajednice (ground truth) koje smo sami definirali
ground_truth = {
    "A": 0, "B": 0, "C": 0,
    "D": 1, "E": 1, "F": 1,
    "G": 2, "H": 2, "I": 2,
}

print(f"Čvorovi: {G_small.number_of_nodes()}, Veze: {G_small.number_of_edges()}")
print(f"Veze unutar zajednica: 9 (3 trokuta × 3 brida)")
print(f"Veze između zajednica: 2 (mostovi C-D i F-G)")

draw_communities(G_small, ground_truth, pos_small,
                 title="Mali graf: 3 zajednice (ground truth) + 2 mosta")
plt.tight_layout()
plt.show()

---

## Korak 3: Modularnost — mjera kvalitete particije

**Što mjerimo:** Modularnost (Q) uspoređuje broj veza unutar grupa s očekivanim brojem u nasumičnoj mreži. Viša Q → bolji "fit" zajednica.

- **Q ≈ 0** → nema strukture zajednica (ili loša podjela).
- **Q > 0.3** → značajna struktura.
- **Q > 0.7** → vrlo izražene zajednice.

Ovdje računamo modularnost za tri scenarija:
1. **Dobra particija** (naše 3 grupe) — trebala bi biti visoka.
2. **Loša particija** (nasumična podjela) — trebala bi biti niska.
3. **Trivijalna particija** (svi u jednoj grupi) — Q = 0.

*(Detaljno objašnjenje: [pogl. 6 — Kako radi modularnost](../content/06_analiza_zajednica.md#kako-radi-modularnost--intuitivno).)*

In [ ]:
def partition_to_communities(partition):
    """Pretvori dict {node: community_id} u listu setova."""
    comms = {}
    for node, cid in partition.items():
        comms.setdefault(cid, set()).add(node)
    return list(comms.values())

# 1. Dobra particija (ground truth)
Q_good = nx.community.modularity(G_small, partition_to_communities(ground_truth))

# 2. Loša particija (proizvoljno pomiješana)
bad_partition = {"A": 0, "B": 1, "C": 2, "D": 0, "E": 1, "F": 2, "G": 0, "H": 1, "I": 2}
Q_bad = nx.community.modularity(G_small, partition_to_communities(bad_partition))

# 3. Trivijalna particija (svi zajedno)
trivial = {n: 0 for n in G_small.nodes()}
Q_trivial = nx.community.modularity(G_small, partition_to_communities(trivial))

print("Modularnost (Q) za različite particije:")
print(f"  Dobra (3 stvarne grupe):   Q = {Q_good:.4f}  ← visoka, zajednice su izražene")
print(f"  Loša (nasumično):          Q = {Q_bad:.4f}  ← niska, podjela ne odgovara strukturi")
print(f"  Trivijalna (svi u jednoj): Q = {Q_trivial:.4f}  ← nula, nema podjele")

# Vizualna usporedba
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
draw_communities(G_small, ground_truth, pos_small, f"Dobra particija\nQ = {Q_good:.3f}", axes[0])
draw_communities(G_small, bad_partition, pos_small, f"Loša particija\nQ = {Q_bad:.3f}", axes[1])
draw_communities(G_small, trivial, pos_small, f"Trivijalna\nQ = {Q_trivial:.3f}", axes[2])
plt.suptitle("Modularnost: dobra vs loša vs trivijalna particija", fontsize=12)
plt.tight_layout()
plt.show()

---

## Korak 4: Louvain algoritam

**Kako radi:** Odozdo prema gore — svaki čvor počinje kao vlastita zajednica; zatim se čvorovi premještaju u zajednicu susjeda koja daje najveći porast modularnosti. Kad se više ništa ne poboljšava, zajednice se komprimiraju u "superčvorove" i postupak se ponavlja.

**Prednosti:** Brz, ne zahtijeva parametar (broj zajednica), radi i na velikim mrežama.
**Nedostatak:** Element slučajnosti — različita pokretanja mogu dati malo različite rezultate.

*(Detaljno: [pogl. 6 — Kako radi Louvain](../content/06_analiza_zajednica.md#kako-radi-louvain--korak-po-korak).)*

In [ ]:
# NetworkX >= 2.7 ima ugrađeni Louvain
louvain_comms = nx.community.louvain_communities(G_small, seed=42)

# Pretvori u {node: id} rječnik
louvain_part = {}
for i, comm in enumerate(louvain_comms):
    for node in comm:
        louvain_part[node] = i

Q_louvain = nx.community.modularity(G_small, louvain_comms)

print(f"Louvain — broj pronađenih zajednica: {len(louvain_comms)}")
for i, comm in enumerate(louvain_comms):
    print(f"  Zajednica {i}: {sorted(comm)}")
print(f"  Modularnost Q = {Q_louvain:.4f}")

# Usporedba s ground truth
match = sum(1 for n in G_small.nodes() if louvain_part[n] == ground_truth[n])
print(f"\nPodudaranje s ground truth: {match}/{G_small.number_of_nodes()} čvorova")
print("(Napomena: ID-evi zajednica mogu biti drugačiji, ali struktura grupa bi trebala biti ista.)")

draw_communities(G_small, louvain_part, pos_small,
                 f"Louvain na malom grafu — Q = {Q_louvain:.3f}")
plt.tight_layout()
plt.show()

---

## Korak 5: Girvan–Newman algoritam

**Kako radi:** Odozgo prema dolje — računa **betweenness** za svaki brid; uklanja brid s najvišim betweennessom; ponavlja dok se mreža ne raspadne na željeni broj komponenti. Svaki korak daje drukčiju particiju.

**Prednosti:** Konceptualno intuitivan — uklanjamo "mostove".
**Nedostatak:** Spor za velike mreže (betweenness se mora preračunati nakon svakog uklanjanja).

*(Detaljno: [pogl. 6 — Kako radi Girvan–Newman](../content/06_analiza_zajednica.md#kako-radi-girvannewman--korak-po-korak).)*

In [ ]:
# Girvan-Newman vraća generator — svaki korak daje jednu particiju više
gn_gen = nx.community.girvan_newman(G_small)

# Prikupimo particije za prvih nekoliko koraka i pratimo modularnost
gn_results = []
for i, communities in enumerate(gn_gen):
    Q = nx.community.modularity(G_small, communities)
    gn_results.append((len(communities), communities, Q))
    if len(communities) >= G_small.number_of_nodes():
        break

print("Girvan–Newman: particije kroz korake uklanjanja bridova\n")
print(f"{'Korak':>5} {'Br. zajednica':>14} {'Q':>8}")
print("-" * 30)
for step, (n_comm, comms, Q) in enumerate(gn_results, 1):
    marker = " ← optimalno" if Q == max(r[2] for r in gn_results) else ""
    print(f"{step:>5} {n_comm:>14} {Q:>8.4f}{marker}")

# Pronađi optimalnu particiju (max Q)
best_idx = max(range(len(gn_results)), key=lambda i: gn_results[i][2])
best_n, best_comms, best_Q = gn_results[best_idx]

gn_part = {}
for i, comm in enumerate(best_comms):
    for node in comm:
        gn_part[node] = i

print(f"\nOptimalna particija: {best_n} zajednice, Q = {best_Q:.4f}")
for i, comm in enumerate(best_comms):
    print(f"  Zajednica {i}: {sorted(comm)}")

In [ ]:
# Vizualizacija: Girvan-Newman kroz korake
steps_to_show = [0, best_idx, len(gn_results) - 1]
steps_to_show = sorted(set(s for s in steps_to_show if s < len(gn_results)))

fig, axes = plt.subplots(1, len(steps_to_show), figsize=(5 * len(steps_to_show), 4))
if len(steps_to_show) == 1:
    axes = [axes]

for ax, idx in zip(axes, steps_to_show):
    n_c, comms, Q = gn_results[idx]
    part = {}
    for i, comm in enumerate(comms):
        for node in comm:
            part[node] = i
    opt = " ★" if idx == best_idx else ""
    draw_communities(G_small, part, pos_small,
                     f"Korak {idx+1}: {n_c} zajednice, Q={Q:.3f}{opt}", ax)

plt.suptitle("Girvan–Newman: uklanjanje bridova korak po korak", fontsize=12)
plt.tight_layout()
plt.show()

---

## Korak 6: Label Propagation

**Kako radi:** Svaki čvor počinje s jedinstvenom oznakom. U svakom koraku preuzima najčešću oznaku svojih susjeda. Postupak se ponavlja dok oznake ne konvergiraju.

**Prednosti:** Iznimno brz, ne zahtijeva parametre.
**Nedostatak:** Rezultat može varirati između pokretanja (nedeterministički).

*(Usporedba algoritama: [pogl. 6 — Usporedba algoritama](../content/06_analiza_zajednica.md#usporedba-algoritama).)*

In [ ]:
lpa_comms = list(nx.community.label_propagation_communities(G_small))
Q_lpa = nx.community.modularity(G_small, lpa_comms)

lpa_part = {}
for i, comm in enumerate(lpa_comms):
    for node in comm:
        lpa_part[node] = i

print(f"Label Propagation — broj zajednica: {len(lpa_comms)}")
for i, comm in enumerate(lpa_comms):
    print(f"  Zajednica {i}: {sorted(comm)}")
print(f"  Modularnost Q = {Q_lpa:.4f}")

draw_communities(G_small, lpa_part, pos_small,
                 f"Label Propagation — Q = {Q_lpa:.3f}")
plt.tight_layout()
plt.show()

---

## Korak 7: Greedy Modularity (pohlepna optimizacija)

**Kako radi:** Počinje sa svakim čvorom kao zasebnom zajednicom; u svakom koraku spaja dvije zajednice čije spajanje daje najveći porast modularnosti. Postupak se ponavlja dok se Q više ne može poboljšati.

Sličan Louvainu, ali konceptualno jednostavniji (nema faze agregacije).

In [ ]:
greedy_comms = list(nx.community.greedy_modularity_communities(G_small))
Q_greedy = nx.community.modularity(G_small, greedy_comms)

greedy_part = {}
for i, comm in enumerate(greedy_comms):
    for node in comm:
        greedy_part[node] = i

print(f"Greedy Modularity — broj zajednica: {len(greedy_comms)}")
for i, comm in enumerate(greedy_comms):
    print(f"  Zajednica {i}: {sorted(comm)}")
print(f"  Modularnost Q = {Q_greedy:.4f}")

draw_communities(G_small, greedy_part, pos_small,
                 f"Greedy Modularity — Q = {Q_greedy:.3f}")
plt.tight_layout()
plt.show()

---

## Korak 8: Usporedba svih algoritama na malom grafu

Sad kad smo pokrenuli četiri algoritma na istom grafu, uspoređujemo ih:
- Koliko zajednica svaki pronalazi?
- Koja je modularnost?
- Podudaraju li se s ground truth?

In [ ]:
results = {
    "Ground truth": (ground_truth, Q_good),
    "Louvain": (louvain_part, Q_louvain),
    "Girvan-Newman": (gn_part, best_Q),
    "Label Propagation": (lpa_part, Q_lpa),
    "Greedy Modularity": (greedy_part, Q_greedy),
}

print(f"{'Algoritam':<22} {'Zajednice':>10} {'Q':>8}")
print("-" * 42)
for name, (part, Q) in results.items():
    n_comm = len(set(part.values()))
    print(f"{name:<22} {n_comm:>10} {Q:>8.4f}")

# Vizualna usporedba svih algoritama
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, (part, Q)) in zip(axes, results.items()):
    draw_communities(G_small, part, pos_small, f"{name}\nQ = {Q:.3f}", ax)
plt.suptitle("Usporedba algoritama na malom grafu (9 čvorova, 3 grupe)", fontsize=13)
plt.tight_layout()
plt.show()

---

## Korak 9: Zacharyjev karate klub — klasičan benchmark

**Priča:** Sociolog Wayne Zachary (1977) promatrao je mali karate klub na sveučilištu. Došlo je do sukoba između instruktora (Mr. Hi, čvor 0) i predsjednika (Officer, čvor 33); klub se raspao na dva dijela. Mreža bilježi tko je s kime prijateljevao.

Ovo je "hello world" detekcije zajednica — algoritam bi trebao pronaći podjelu sličnu stvarnom raskolu.

*(Primjer opisan u [pogl. 6 — Zacharyjev karate klub](../content/06_analiza_zajednica.md#primjeri-iz-stvarnog-života-istraživanja).)*

In [ ]:
G_karate = nx.karate_club_graph()
pos_karate = nx.spring_layout(G_karate, seed=42)

# Ground truth: NetworkX pohranjuje stvarnu podjelu u atributu 'club'
gt_karate = {}
club_map = {"Mr. Hi": 0, "Officer": 1}
for node in G_karate.nodes():
    gt_karate[node] = club_map[G_karate.nodes[node]["club"]]

Q_gt_karate = nx.community.modularity(
    G_karate, partition_to_communities(gt_karate)
)

print(f"Zacharyjev karate klub: {G_karate.number_of_nodes()} čvorova, "
      f"{G_karate.number_of_edges()} veza")
print(f"Stvarna podjela: Mr. Hi ({sum(1 for v in gt_karate.values() if v==0)} članova) "
      f"vs Officer ({sum(1 for v in gt_karate.values() if v==1)} članova)")
print(f"Modularnost stvarne podjele: Q = {Q_gt_karate:.4f}")

draw_communities(G_karate, gt_karate, pos_karate,
                 f"Zachary karate klub — stvarna podjela (Q = {Q_gt_karate:.3f})")
plt.tight_layout()
plt.show()

---

## Korak 10: Svi algoritmi na karate klubu

Primjenjujemo iste četiri algoritma i uspoređujemo s poznatom podjelom.

In [ ]:
# Louvain
kc_louvain = nx.community.louvain_communities(G_karate, seed=42)
kc_louvain_part = {}
for i, c in enumerate(kc_louvain):
    for n in c:
        kc_louvain_part[n] = i
Q_kc_louv = nx.community.modularity(G_karate, kc_louvain)

# Girvan-Newman (tražimo particiju s max Q)
kc_gn_gen = nx.community.girvan_newman(G_karate)
kc_gn_best = None
kc_gn_best_Q = -1
for comms in kc_gn_gen:
    Q = nx.community.modularity(G_karate, comms)
    if Q > kc_gn_best_Q:
        kc_gn_best_Q = Q
        kc_gn_best = comms
    if len(comms) > 10:
        break
kc_gn_part = {}
for i, c in enumerate(kc_gn_best):
    for n in c:
        kc_gn_part[n] = i

# Label Propagation
kc_lpa = list(nx.community.label_propagation_communities(G_karate))
kc_lpa_part = {}
for i, c in enumerate(kc_lpa):
    for n in c:
        kc_lpa_part[n] = i
Q_kc_lpa = nx.community.modularity(G_karate, kc_lpa)

# Greedy Modularity
kc_greedy = list(nx.community.greedy_modularity_communities(G_karate))
kc_greedy_part = {}
for i, c in enumerate(kc_greedy):
    for n in c:
        kc_greedy_part[n] = i
Q_kc_greedy = nx.community.modularity(G_karate, kc_greedy)

kc_results = {
    "Stvarna podjela": (gt_karate, Q_gt_karate),
    "Louvain": (kc_louvain_part, Q_kc_louv),
    "Girvan-Newman": (kc_gn_part, kc_gn_best_Q),
    "Label Propagation": (kc_lpa_part, Q_kc_lpa),
    "Greedy Modularity": (kc_greedy_part, Q_kc_greedy),
}

print(f"{'Algoritam':<22} {'Zajednice':>10} {'Q':>8}")
print("-" * 42)
for name, (part, Q) in kc_results.items():
    n_comm = len(set(part.values()))
    print(f"{name:<22} {n_comm:>10} {Q:>8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
for ax, (name, (part, Q)) in zip(axes, kc_results.items()):
    n_c = len(set(part.values()))
    draw_communities(G_karate, part, pos_karate,
                     f"{name}\n{n_c} zajednica, Q = {Q:.3f}", ax)
plt.suptitle("Zacharyjev karate klub — usporedba algoritama", fontsize=13)
plt.tight_layout()
plt.show()

---

## Korak 11: Analiza mostova i graničnih čvorova

Zajednice su zanimljive, ali jednako su zanimljivi **čvorovi na granici** — oni koji imaju veze i u svoju i u susjednu zajednicu. To su **brokeri** ili **mostovi** (u [pogl. 6](../content/06_analiza_zajednica.md) primjeri: Tyrion u GoT, Post Malone između žanrova, Snape u Hogwartsu).

Identificiramo ih tako da gledamo: koliki udio veza čvora ide prema njegovoj zajednici, a koliki prema drugim zajednicama?

In [ ]:
# Koristimo Louvain particiju na karate klubu
partition = kc_louvain_part

bridge_scores = []
for node in G_karate.nodes():
    my_comm = partition[node]
    neighbors = list(G_karate.neighbors(node))
    if len(neighbors) == 0:
        continue
    same = sum(1 for nb in neighbors if partition[nb] == my_comm)
    diff = len(neighbors) - same
    ratio = diff / len(neighbors)
    bridge_scores.append((node, ratio, same, diff, len(neighbors)))

bridge_scores.sort(key=lambda x: -x[1])

print("Top 10 'mostova' — čvorovi s najvišim udjelom veza prema drugim zajednicama:\n")
print(f"{'Čvor':>5} {'Udio izvana':>12} {'Unutar':>7} {'Izvana':>7} {'Ukupno':>8}")
print("-" * 42)
for node, ratio, same, diff, total in bridge_scores[:10]:
    print(f"{node:>5} {ratio:>12.1%} {same:>7} {diff:>7} {total:>8}")

# Vizualizacija: veličina čvora = bridge score
bridge_dict = {node: ratio for node, ratio, *_ in bridge_scores}
node_sizes = [bridge_dict.get(n, 0) * 1500 + 200 for n in G_karate.nodes()]

fig, ax = plt.subplots(figsize=(8, 6))
colors = [PALETTE[partition[n] % len(PALETTE)] for n in G_karate.nodes()]
nx.draw(G_karate, pos_karate, ax=ax, node_color=colors, node_size=node_sizes,
        with_labels=True, font_size=7, edge_color="#cccccc", width=0.8)
ax.set_title("Mostovi: veličina čvora ∝ udio veza prema drugim zajednicama", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 12: Stabilnost — ponovljena pokretanja Louvainja

Louvain uključuje element slučajnosti. Ponovimo ga 20 puta i pogledajmo:
- Koliko puta dobijemo isti broj zajednica?
- Koliko varira modularnost?

Ako su rezultati stabilni, možemo im vjerovati. Ako jako variraju, struktura zajednica možda nije jasno definirana.

*(Problem stabilnosti: [pogl. 6 — Problemi i izazovi](../content/06_analiza_zajednica.md#problemi-i-izazovi).)*

In [ ]:
n_runs = 20
q_values = []
n_communities = []

for seed in range(n_runs):
    comms = nx.community.louvain_communities(G_karate, seed=seed)
    Q = nx.community.modularity(G_karate, comms)
    q_values.append(Q)
    n_communities.append(len(comms))

print(f"Louvain ponovljen {n_runs}× na karate klubu:\n")
print(f"Modularnost — min: {min(q_values):.4f}, max: {max(q_values):.4f}, "
      f"prosjek: {sum(q_values)/len(q_values):.4f}")
print(f"Broj zajednica — distribucija: {dict(Counter(n_communities))}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(range(n_runs), q_values, color="steelblue", alpha=0.8)
ax1.axhline(sum(q_values)/len(q_values), color="red", ls="--", label="prosjek")
ax1.set_xlabel("Pokretanje")
ax1.set_ylabel("Modularnost Q")
ax1.set_title("Modularnost kroz 20 pokretanja")
ax1.legend()

comm_counts = Counter(n_communities)
ax2.bar(comm_counts.keys(), comm_counts.values(), color="coral", alpha=0.8)
ax2.set_xlabel("Broj zajednica")
ax2.set_ylabel("Koliko puta")
ax2.set_title("Distribucija broja zajednica")

plt.suptitle("Stabilnost Louvainja: 20 pokretanja na karate klubu", fontsize=12)
plt.tight_layout()
plt.show()

---

## Korak 13: Profil zajednica — tko je u kojoj grupi?

Kad imamo particiju, korisno je pogledati **profil** svake zajednice: koliko članova, koji su ključni čvorovi (po degree, betweenness), i koliko je zajednica interno gusta.

*(Savjeti za interpretaciju: [pogl. 6 — Interpretacija rezultata](../content/06_analiza_zajednica.md#interpretacija-rezultata--praktični-savjeti).)*

In [ ]:
partition = kc_louvain_part
bet = nx.betweenness_centrality(G_karate)
deg = dict(G_karate.degree())

communities_dict = {}
for node, cid in partition.items():
    communities_dict.setdefault(cid, []).append(node)

print("Profil zajednica (Louvain na karate klubu):\n")
for cid in sorted(communities_dict):
    members = communities_dict[cid]
    subgraph = G_karate.subgraph(members)
    density = nx.density(subgraph)
    top_degree = max(members, key=lambda n: deg[n])
    top_between = max(members, key=lambda n: bet[n])

    print(f"Zajednica {cid}:")
    print(f"  Članovi ({len(members)}): {sorted(members)}")
    print(f"  Interna gustoća: {density:.3f}")
    print(f"  Čvor s najvišim degree: {top_degree} (degree={deg[top_degree]})")
    print(f"  Čvor s najvišim betweenness: {top_between} (bet={bet[top_between]:.3f})")
    print()

---

## Korak 14: Vlastiti primjer — mreža "Game of Thrones"

Napravimo malu GoT mrežu (pojednostavljenu) i vidimo hoće li algoritmi detektirati kuće.

*(Konceptualno: [pogl. 6 — GoT kuće kao zajednice](../content/06_analiza_zajednica.md#primjeri-iz-filmova-i-serija).)*

In [ ]:
G_got = nx.Graph()

# Starki
G_got.add_edges_from([
    ("Ned", "Catelyn"), ("Ned", "Robb"), ("Ned", "Arya"),
    ("Ned", "Sansa"), ("Catelyn", "Robb"), ("Robb", "Jon"),
    ("Arya", "Jon"), ("Sansa", "Catelyn"),
])

# Lanisteri
G_got.add_edges_from([
    ("Cersei", "Jaime"), ("Cersei", "Tywin"), ("Jaime", "Tywin"),
    ("Cersei", "Joffrey"), ("Tywin", "Tyrion"),
])

# Targaryeni
G_got.add_edges_from([
    ("Daenerys", "Jorah"), ("Daenerys", "Missandei"),
    ("Daenerys", "Grey Worm"), ("Jorah", "Missandei"),
])

# Mostovi između kuća
G_got.add_edges_from([
    ("Ned", "Cersei"),      # Stark ↔ Lannister
    ("Sansa", "Joffrey"),   # Stark ↔ Lannister
    ("Tyrion", "Daenerys"), # Lannister ↔ Targaryen
    ("Jon", "Daenerys"),    # Stark ↔ Targaryen
])

# Ground truth: kuće
got_houses = {
    "Ned": 0, "Catelyn": 0, "Robb": 0, "Arya": 0, "Sansa": 0, "Jon": 0,
    "Cersei": 1, "Jaime": 1, "Tywin": 1, "Joffrey": 1, "Tyrion": 1,
    "Daenerys": 2, "Jorah": 2, "Missandei": 2, "Grey Worm": 2,
}
house_names = {0: "Stark", 1: "Lannister", 2: "Targaryen"}

pos_got = nx.spring_layout(G_got, seed=7)

print(f"GoT mreža: {G_got.number_of_nodes()} likova, {G_got.number_of_edges()} veza")
for hid, hname in house_names.items():
    members = [n for n, h in got_houses.items() if h == hid]
    print(f"  {hname}: {members}")

In [ ]:
# Louvain na GoT mreži
got_louvain = nx.community.louvain_communities(G_got, seed=42)
got_louv_part = {}
for i, c in enumerate(got_louvain):
    for n in c:
        got_louv_part[n] = i
Q_got_gt = nx.community.modularity(G_got, partition_to_communities(got_houses))
Q_got_louv = nx.community.modularity(G_got, got_louvain)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
draw_communities(G_got, got_houses, pos_got,
                 f"GoT — stvarne kuće (Q = {Q_got_gt:.3f})", axes[0])
draw_communities(G_got, got_louv_part, pos_got,
                 f"GoT — Louvain (Q = {Q_got_louv:.3f})", axes[1])
plt.suptitle("Game of Thrones: detektira li algoritam kuće?", fontsize=13)
plt.tight_layout()
plt.show()

print("\nLouvain zajednice:")
for i, comm in enumerate(got_louvain):
    print(f"  Zajednica {i}: {sorted(comm)}")
print(f"\nModularnost — kuće: {Q_got_gt:.4f}, Louvain: {Q_got_louv:.4f}")

---

## Korak 15: Vlastiti primjer — mreža glazbenih suradnji

Mreža glazbenika: tko s kime surađuje? Žanrovi bi trebali formirati zajednice; crossover umjetnici su mostovi.

*(Konceptualno: [pogl. 6 — Glazbeni žanrovi kao zajednice](../content/06_analiza_zajednica.md#primjeri-iz-umjetnosti-književnosti-i-glazbe).)*

In [ ]:
G_music = nx.Graph()

# Hip-hop klaster
G_music.add_edges_from([
    ("Kendrick", "Drake"), ("Kendrick", "J. Cole"),
    ("Drake", "21 Savage"), ("Drake", "J. Cole"),
    ("21 Savage", "J. Cole"),
])

# Pop klaster
G_music.add_edges_from([
    ("Taylor Swift", "Ed Sheeran"), ("Taylor Swift", "Ariana Grande"),
    ("Ed Sheeran", "Ariana Grande"), ("Taylor Swift", "Selena Gomez"),
    ("Ariana Grande", "Selena Gomez"),
])

# Rock/Alt klaster
G_music.add_edges_from([
    ("Foo Fighters", "Queens of SA"), ("Foo Fighters", "Arctic Monkeys"),
    ("Queens of SA", "Arctic Monkeys"),
])

# Mostovi (crossover)
G_music.add_edges_from([
    ("Post Malone", "Drake"),          # hip-hop ↔ crossover
    ("Post Malone", "Taylor Swift"),    # crossover ↔ pop
    ("Post Malone", "Foo Fighters"),    # crossover ↔ rock
])

genre_truth = {
    "Kendrick": 0, "Drake": 0, "J. Cole": 0, "21 Savage": 0,
    "Taylor Swift": 1, "Ed Sheeran": 1, "Ariana Grande": 1, "Selena Gomez": 1,
    "Foo Fighters": 2, "Queens of SA": 2, "Arctic Monkeys": 2,
    "Post Malone": 3,
}

pos_music = nx.spring_layout(G_music, seed=15)

# Louvain
music_louv = nx.community.louvain_communities(G_music, seed=42)
music_louv_part = {}
for i, c in enumerate(music_louv):
    for n in c:
        music_louv_part[n] = i
Q_music = nx.community.modularity(G_music, music_louv)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
draw_communities(G_music, genre_truth, pos_music,
                 "Glazba — žanrovi (ručna podjela)", axes[0])
draw_communities(G_music, music_louv_part, pos_music,
                 f"Glazba — Louvain (Q = {Q_music:.3f})", axes[1])
plt.suptitle("Glazbene suradnje: žanrovi kao zajednice, crossover kao most", fontsize=13)
plt.tight_layout()
plt.show()

print("Louvain zajednice:")
for i, comm in enumerate(music_louv):
    print(f"  Zajednica {i}: {sorted(comm)}")

# Tko je most?
bet_music = nx.betweenness_centrality(G_music)
print(f"\nNajviši betweenness (most između žanrova):")
for n, b in sorted(bet_music.items(), key=lambda x: -x[1])[:3]:
    print(f"  {n}: {b:.3f}")

---

## Korak 16: Modularnost kao funkcija broja zajednica

Koliko zajednica je "pravo"? Jedan način: pratimo modularnost dok Girvan–Newman postupno rastavlja mrežu. Q najprije raste (uklanjamo mostove, grupe se razdvajaju), a zatim pada (previše fragmentacija). **Vrh Q** sugerira optimalan broj zajednica.

In [ ]:
gn_karate = nx.community.girvan_newman(G_karate)
q_by_k = []

for comms in gn_karate:
    k = len(comms)
    Q = nx.community.modularity(G_karate, comms)
    q_by_k.append((k, Q))
    if k >= 15:
        break

ks = [x[0] for x in q_by_k]
qs = [x[1] for x in q_by_k]
best_k = ks[qs.index(max(qs))]

plt.figure(figsize=(8, 4))
plt.plot(ks, qs, "o-", color="steelblue", markersize=6)
plt.axvline(best_k, color="red", ls="--", alpha=0.7, label=f"Optimalno: k={best_k}")
plt.xlabel("Broj zajednica (k)")
plt.ylabel("Modularnost (Q)")
plt.title("Girvan–Newman na karate klubu: Q vs broj zajednica")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimalan broj zajednica po max Q: k = {best_k} (Q = {max(qs):.4f})")

---

## Korak 17: Sažetak svih mjera u jednoj tablici

Za svaki algoritam i svaki dataset prikazujemo: broj zajednica, modularnost, i ključne karakteristike.

In [ ]:
summary_data = []

# Mali graf
for name, (part, Q) in results.items():
    n_c = len(set(part.values()))
    summary_data.append({"Dataset": "Mali graf (9 čv.)", "Algoritam": name,
                         "Br. zajednica": n_c, "Q": round(Q, 4)})

# Karate klub
for name, (part, Q) in kc_results.items():
    n_c = len(set(part.values()))
    summary_data.append({"Dataset": "Karate klub (34 čv.)", "Algoritam": name,
                         "Br. zajednica": n_c, "Q": round(Q, 4)})

df_summary = pd.DataFrame(summary_data)
print("Sažetak svih rezultata:\n")
print(df_summary.to_string(index=False))

---

## Zaključak

**Što smo naučili:**

1. **Modularnost (Q)** mjeri kvalitetu particije — uspoređuje stvarne veze unutar grupa s očekivanim u nasumičnoj mreži. Q > 0.3 znači da zajednice postoje.

2. **Louvain** je brz, ne zahtijeva parametre i obično daje dobre rezultate. Na karate klubu pronalazi 3–4 zajednice s Q ≈ 0.41–0.42.

3. **Girvan–Newman** postupno rastavlja mrežu uklanjanjem mostova. Daje dendogram — možemo birati razinu. Spor za velike mreže.

4. **Label Propagation** je najbrži, ali nedeterministički — rezultati mogu varirati.

5. **Greedy Modularity** je konceptualno najjednostavniji pristup maksimizaciji Q.

6. **Mostovi** (čvorovi na granici zajednica) su posebno zanimljivi — to su brokeri, posrednici, kulturni prevodioci.

7. **Stabilnost** treba provjeriti ponovljenim pokretanjima.

8. **Interpretacija** je najvažniji korak — algoritam daje particiju, ali **smisao** moramo dati mi.

**Sadržaj ↔ kod:** Za definicije, intuiciju, konceptualne primjere (GoT, MCU, glazba, Twitter, Reddit) i savjete za interpretaciju vidi [06_analiza_zajednica.md](../content/06_analiza_zajednica.md). Tablica "Povezivanje sadržaj ↔ kod" na kraju tog dokumenta usmjeruje na odgovarajuće korake u ovoj bilježnici.